# 1. O DBSCAN klasteriranju

DBSCAN je algoritam klasteriranja baziran na gustoći.

Za razliku od K-Means algoritma, DBSCAN ne zahtijeva unaprijed zadavanje broja klastera. Umjesto toga koristi dva parametra:

- `eps` — radijus susjedstva,
- `min_samples` — minimalan broj točaka potreban da bi se točka smatrala jezgrenom točkom.

DBSCAN može identificirati klastere proizvoljnog oblika te označiti šum, odnosno točke koje ne pripadaju nijednom klasteru.

Radimo ga na slučajnom uzorku.

In [15]:
df = pd.read_csv("/airbnb_features.csv")

## 2. Učitavanje podataka, odabir atributa, uzorkovanje

Za analizu se koristi skup podataka nakon prethodno provedenog feature engineeringa. Taj skup već sadrži dodatno konstruirane značajke poput udaljenosti od centra, omjera dostupnosti i mjere popularnosti smještaja.

Za DBSCAN su odabrani atributi koji opisuju cijenu, lokaciju, popularnost, dostupnost i ocjenu smještaja.

Korišteni atributi su:

- `price`
- `distance_to_center`
- `popularity_score`
- `availability_ratio`
- `review_rate_number`

Ovi atributi omogućuju analizu sličnosti smještaja prema ekonomskim, lokacijskim i korisničkim karakteristikama.

Budući da DBSCAN može biti računalno zahtjevan na većim skupovima podataka, analiza je provedena na slučajnom uzorku od 10 000 smještaja.

Korištenjem `random_state = 42` osigurana je ponovljivost rezultata.

In [30]:
df_dbscan = df.sample(
    n=10000,
    random_state=42
).copy()

dbscan_features = [
    "price",
    "distance_to_center",
    "popularity_score",
    "availability_ratio",
    "review_rate_number"
]

X_dbscan = df_dbscan[dbscan_features].fillna(0)

## 3. Skaliranje podataka

DBSCAN koristi udaljenosti između točaka, pa je potrebno standardizirati podatke prije primjene algoritma.

Bez skaliranja bi atributi s većim numeričkim vrijednostima, primjerice cijena, imali nerazmjerno velik utjecaj na rezultat klasteriranja.

Standardizacijom se svaki atribut transformira tako da ima sredinu 0 i standardnu devijaciju 1.

In [31]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_dbscan_scaled = scaler.fit_transform(X_dbscan)

## 4. Primjena DBSCAN algoritma

DBSCAN koristi dva ključna parametra:

- `eps`: radijus susjedstva oko svake točke,
- `min_samples`: minimalan broj točaka potreban da bi se neka točka smatrala jezgrenom točkom.

U ovoj analizi korišteni su parametri `eps = 0.8` i `min_samples = 10`.

Oznaka `-1` u rezultatima označava šum, odnosno smještaje koji nisu pridruženi nijednom klasteru.

In [32]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(
    eps=0.8,
    min_samples=10
)

df_dbscan["dbscan_cluster"] = dbscan.fit_predict(X_dbscan_scaled)

df_dbscan["dbscan_cluster"].value_counts().sort_index()

,count
dbscan_cluster,
-1,563
0,9437


In [33]:
dbscan_summary = df_dbscan.groupby("dbscan_cluster")[dbscan_features].mean()
dbscan_summary

,price,distance_to_center,popularity_score,availability_ratio,review_rate_number
dbscan_cluster,,,,,
-1,523.550622,13.429449,3.745152,0.502124,2.912966
0,523.255060,7.104532,-0.202624,0.369547,3.347038


## 5. Interpretacija
Primjenom DBSCAN algoritma na uzorku od 10 000 smještaja identificiran je jedan dominantan klaster te skup od 563 objekta označenih kao šum (oznaka -1).
Analiza prosječnih vrijednosti pokazuje da se smještaji označeni kao šum ne razlikuju značajno prema cijeni, ali se nalaze znatno dalje od centra grada te imaju veći popularity_score. Takvi objekti predstavljaju netipične smještaje koji odstupaju od većine promatranih oglasa.
Rezultati sugeriraju da su geografska lokacija i obrasci popularnosti važniji za izdvajanje anomalnih smještaja od same cijene.

# 6. Zaključak

Primjenom DBSCAN algoritma provedena je dodatna analiza strukture Airbnb podataka.

Za razliku od K-Means algoritma, DBSCAN ne zahtijeva unaprijed zadavanje broja klastera te može detektirati šum i netipične objekte.

Na uzorku od 10 000 smještaja algoritam je identificirao jedan dominantan klaster i manji broj anomalnih smještaja.

Dobiveni rezultati pokazuju da se netipični smještaji najviše razlikuju prema udaljenosti od centra i popularnosti, dok cijena nije bila presudan faktor razlikovanja.